# Alfido Tech — Task 4: Website Traffic & User Journey Analysis

**Dataset:** `traffic.csv` (2021-08-19 to 2021-08-25)  
**Objective:** Analyze traffic activity, engagement, top content/link entries, and identify conversion optimizations.

> **Important data limitation:** This file contains event type, date, geography, music metadata and `linkid`, but **no timestamp, user/session ID, referrer/UTM source, page URL/landing-page field, exit page, or session duration**. Therefore true sessions, unique users, bounce rate, average session duration, referral-source performance, and ordered user journeys cannot be computed reliably. This notebook reports only metrics supported by the supplied data and explicitly flags the missing instrumentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Update this path if the CSV is stored elsewhere
DATA_PATH = "traffic.csv"
df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
for c in ["country","city","artist","album","track","isrc","linkid"]:
    df[c] = df[c].fillna("Unknown")

df.info()
df.head()

## 1. Data quality and cleaning

In [ ]:
print("Rows:", len(df))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Unique link IDs:", df["linkid"].nunique())
print("\nEvent counts:")
print(df["event"].value_counts())
print("\nMissing values after cleaning:")
print(df.isna().sum().sort_values(ascending=False).head(10))

## 2. Traffic and engagement metrics

In [ ]:
total_events = len(df)
pageviews = (df["event"] == "pageview").sum()
clicks = (df["event"] == "click").sum()
previews = (df["event"] == "preview").sum()

metrics = pd.Series({
    "Total events": total_events,
    "Pageviews": pageviews,
    "Clicks": clicks,
    "Previews": previews,
    "Click / pageview proxy": clicks / pageviews if pageviews else np.nan,
    "Preview / pageview proxy": previews / pageviews if pageviews else np.nan,
    "Unique link IDs (content proxy)": df["linkid"].nunique(),
    "Unique users": np.nan,
    "Sessions": np.nan,
    "Bounce rate": np.nan,
    "Average session duration": np.nan
})
metrics

### Metric interpretation

The click/pageview and preview/pageview ratios are **event-level engagement proxies**, not conversion rates. Unique users and sessions cannot be calculated because the dataset has no user/session identifier. Bounce rate requires session boundaries and a definition of a bounced visit; average session duration requires event timestamps.

## 3. Daily traffic trend

In [ ]:
daily = df.groupby(["date","event"]).size().unstack(fill_value=0)
ax = daily.plot(figsize=(10,5), marker="o")
ax.set_title("Daily Event Volume")
ax.set_xlabel("Date")
ax.set_ylabel("Events")
plt.tight_layout()
plt.show()

## 4. Top entry/content proxies

In [ ]:
top_links = (df.groupby("linkid")
    .agg(events=("event","size"),
         pageviews=("event", lambda x: (x=="pageview").sum()),
         previews=("event", lambda x: (x=="preview").sum()),
         clicks=("event", lambda x: (x=="click").sum()))
    .sort_values("events", ascending=False)
    .head(10))
top_links

In [ ]:
top_tracks = df["track"].value_counts().head(10)
top_tracks.sort_values().plot(kind="barh", figsize=(9,5))
plt.title("Top Tracks by Event Volume")
plt.xlabel("Events")
plt.tight_layout()
plt.show()

## 5. Geography and referral-source limitation

In [ ]:
top_countries = df["country"].value_counts().head(15)
top_countries.sort_values().plot(kind="barh", figsize=(9,6))
plt.title("Top Countries by Event Volume")
plt.xlabel("Events")
plt.tight_layout()
plt.show()

print("Referral source / UTM analysis: NOT AVAILABLE — no referrer, source, medium, campaign, or UTM columns are present.")

## 6. User-flow limitation

In [ ]:
# A real flow requires ordered events per user/session.
# The dataset has no user_id/session_id and no event timestamp.
journey_status = {
    "Ordered user journeys": "Not computable",
    "Top landing pages": "Not directly computable; linkid is a content/link identifier",
    "Top exit pages": "Not computable",
    "Bounce rate": "Not computable",
    "Average session duration": "Not computable",
    "Referral sources": "Not available"
}
pd.Series(journey_status)

## 7. Five conversion optimizations for Alfido Tech

1. **Implement session and user tracking:** add anonymous `user_id`/`session_id` plus event timestamps so sessions, users, bounce rate and duration can be measured.
2. **Capture acquisition data:** record referrer, UTM source/medium/campaign and landing URL to identify channels that generate engaged visitors and conversions.
3. **Instrument a conversion funnel:** track landing → key-content view → CTA click → form/contact/demo completion, with a unique conversion event.
4. **Optimize high-traffic entry content:** use the highest-volume `linkid`/content items as priority pages for stronger CTAs, clearer value propositions and relevant next-step links.
5. **Create an exit/engagement dashboard:** once page and session IDs are available, identify high-exit pages and short sessions, then A/B test CTA placement, page speed, navigation and forms.

## Executive takeaway

The supplied log contains **226,278 events**, including **142,015 pageviews, 55,732 clicks and 28,531 previews**. Clicks represent about **39.2% of pageviews** as an event-level proxy, while previews represent about **20.1%**. However, the requested website KPIs—sessions, users, bounce rate, average session duration, landing/exit pages and referral sources—require fields that are not present. The correct business action is therefore to improve analytics instrumentation first, while using the available event/content/geography data to prioritize engagement optimization.